# Hidden Value: Weekly Report (Word document)

**Goal:** turn the saved projection files into the Word write-up, with no hand editing.

Run this **after** `Hidden_Value_Points_Projection` (required) and `Hidden_Value_Points_Projection_advanced` (optional). It reads the files those two notebooks saved, pulls the current betting lines from the schedule file, and writes `Hidden_Value_WeekNN_Projections.docx` into the same folder.

It never changes a projection. If the saved files change, run this again and the document is rebuilt.

Two things from the hand-built Week 5 document are not in here, because nflreadpy does not carry them: the Kalshi column and the table of other sportsbooks.

## 1. Settings

In [6]:
import os, glob, re, math
import pandas as pd, numpy as np
import nflreadpy as nfl

try:
    import docx
except ImportError:                      # python-docx is not part of a default Anaconda install
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    import docx
from docx.shared import Pt, RGBColor, Inches
from docx.enum.section import WD_ORIENT, WD_SECTION
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

SEASON = 2026
FOLDER = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football"

WEEK = None      # None = the most recent week with a saved projection file. Or set a number, e.g. WEEK = 5

# Anything typed here is printed in the document under "Notes for this week". One string per bullet.
NOTES = [
    # "Eagles at Jaguars is in London and is treated as a neutral site.",
    # "Washington: Jayden Daniels named the starter on Friday.",
]

# Fixed numbers from the 2021-2025 backtest. They only change if the model is re-fitted.
COEF = {"const": 4.41, "off": 6.22, "deff": 2.47, "home": 1.15, "new_qb": -3.38}
K_WIN = 0.1458           # win chance per point of projected margin
BACKTEST = [("League average for every team", "7.96", "10.75", "n/a"),
            ("Points for and points against", "7.59", "10.62", "62%"),
            ("Drive model with no quarterback information", "7.57", "10.54", "60%"),
            ("Base model", "7.50", "10.49", "62%"),
            ("Advanced (QB history, injuries, wind)", "7.42", "10.37", "63%"),
            ("Advanced + market starting point", "7.38", "10.37", "63%"),
            ("Betting market", "7.25", "10.25", "68%")]

NICK = {"ARI": "Cardinals", "ATL": "Falcons", "BAL": "Ravens", "BUF": "Bills", "CAR": "Panthers", "CHI": "Bears", "CIN": "Bengals",
        "CLE": "Browns", "DAL": "Cowboys", "DEN": "Broncos", "DET": "Lions", "GB": "Packers", "HOU": "Texans", "IND": "Colts",
        "JAX": "Jaguars", "KC": "Chiefs", "LA": "Rams", "LAC": "Chargers", "LV": "Raiders", "MIA": "Dolphins", "MIN": "Vikings",
        "NE": "Patriots", "NO": "Saints", "NYG": "Giants", "NYJ": "Jets", "PHI": "Eagles", "PIT": "Steelers", "SEA": "Seahawks",
        "SF": "49ers", "TB": "Buccaneers", "TEN": "Titans", "WAS": "Commanders"}

## 2. Load the saved projections and the betting lines

In [7]:
def to_pd(x):
    try: return x.to_pandas()
    except AttributeError: return x

def week_of(path):
    return int(re.search(r"week(\d+)\.csv$", path).group(1))

base_files = {week_of(p): p for p in glob.glob(os.path.join(FOLDER, "hidden_value_projections_week*.csv"))}
adv_files = {week_of(p): p for p in glob.glob(os.path.join(FOLDER, "hidden_value_projections_advanced_week*.csv"))}
if not base_files:
    raise FileNotFoundError("No saved projections found. Run Hidden_Value_Points_Projection first.")
if WEEK is None:
    WEEK = max(base_files)
if WEEK not in base_files:
    raise FileNotFoundError(f"No saved projection file for week {WEEK}. Saved weeks: {sorted(base_files)}")

P = pd.read_csv(base_files[WEEK])
A = pd.read_csv(adv_files[WEEK]) if WEEK in adv_files else None

sched = to_pd(nfl.load_schedules(seasons=[SEASON]))
sched = sched[sched["game_type"] == "REG"].copy()
S = sched.set_index("game_id")

print(f"week {WEEK}: {len(P)} games in the saved file (saved {P['saved_at'].iloc[0]})",
      "| advanced file:", "found" if A is not None else "not found, the report will leave that part out")

week 9: 15 games in the saved file (saved 2026-11-09 10:31) | advanced file: found


## 3. Build the numbers

Every row is read from the side of the betting favorite, so the model and the line can be compared number for number.

In [8]:
def win_chance(margin):
    return 1 / (1 + math.exp(-K_WIN * margin))

def ml_prob(ml):
    return (-ml) / (-ml + 100) if ml < 0 else 100 / (ml + 100)

def fair_ml(p):
    return -100 * p / (1 - p) if p >= 0.5 else 100 * (1 - p) / p

rows = []
for r in P.itertuples():
    s = S.loc[r.game_id]
    hm, tot = float(s["spread_line"]), float(s["total_line"])        # hm > 0 means the home team is favored
    fav_home = hm >= 0
    fav, dog = (r.home, r.away) if fav_home else (r.away, r.home)
    fav_ml = s["home_moneyline"] if fav_home else s["away_moneyline"]
    dog_ml = s["away_moneyline"] if fav_home else s["home_moneyline"]
    line_p = np.nan
    if pd.notna(fav_ml) and pd.notna(dog_ml):
        a, b = ml_prob(float(fav_ml)), ml_prob(float(dog_ml))
        line_p = a / (a + b)                                           # sportsbook's cut removed
    fav_margin = r.home_margin if fav_home else -r.home_margin         # model's margin for the favorite
    rows.append({"game_id": r.game_id, "away": r.away, "home": r.home, "away_qb": r.away_qb, "home_qb": r.home_qb,
                 "away_pts": r.away_pts, "home_pts": r.home_pts, "margin": r.home_margin, "total": r.total,
                 "played_when_saved": bool(r.already_played),
                 "fav": fav, "dog": dog, "line_spread": abs(hm), "line_total": tot, "line_fav_ml": fav_ml, "line_p": line_p,
                 "line_home": tot / 2 + hm / 2, "line_away": tot / 2 - hm / 2,
                 "fav_margin": fav_margin, "model_p": win_chance(fav_margin), "same_side": fav_margin > 0,
                 "margin_gap": r.home_margin - hm, "total_gap": r.total - tot,
                 "home_score": s["home_score"], "away_score": s["away_score"]})
G = pd.DataFrame(rows)
G["model_fair_ml"] = G["model_p"].map(fair_ml)
open_games = G[~G["played_when_saved"]]

print(f"{len(open_games)} games projected in advance | model and line on different sides in {int((~open_games['same_side']).sum())}")
print(f"average total: model {open_games['total'].mean():.1f}, line {open_games['line_total'].mean():.1f}")

1 games projected in advance | model and line on different sides in 0
average total: model 49.2, line 44.5


## 4. Grade every week that has final scores

Same rule as the projection notebooks: a game that already had a final score when it was saved is not graded.

In [9]:
def grade_all():
    out = []
    for wk in sorted(base_files):
        p = pd.read_csv(base_files[wk])
        p = p[~p["already_played"].astype(bool)]
        a = pd.read_csv(adv_files[wk]) if wk in adv_files else None
        for r in p.itertuples():
            s = S.loc[r.game_id]
            if pd.isna(s["home_score"]):
                continue
            for team, pts, actual, mk in [(r.home, r.home_pts, s["home_score"], r.mkt_total / 2 + r.mkt_home_margin / 2),
                                          (r.away, r.away_pts, s["away_score"], r.mkt_total / 2 - r.mkt_home_margin / 2)]:
                row = {"week": wk, "game_id": r.game_id, "team": team, "is_home": team == r.home, "actual": actual, "Base model": pts, "Betting line": mk}
                if a is not None:
                    m = a[(a["game_id"] == r.game_id) & (a["team"] == team)]
                    if len(m):
                        row["Advanced"], row["Advanced + market"] = m["adv"].iloc[0], m["adv_mkt"].iloc[0]
                out.append(row)
    return pd.DataFrame(out)

graded = grade_all()
MODELS = [m for m in ["Base model", "Advanced", "Advanced + market", "Betting line"] if len(graded) and m in graded.columns]
if len(graded):
    print(f"graded so far: {len(graded)} team scores, weeks {sorted(graded['week'].unique().tolist())}")
    print(graded[MODELS].sub(graded["actual"], axis=0).abs().mean().round(2).to_string())
else:
    print("nothing graded yet")

graded so far: 84 team scores, weeks [5, 6, 7, 8]
Base model           7.78
Advanced             7.60
Advanced + market    7.56
Betting line         7.29


## 5. Write the Word document

In [10]:
INK, MUTED, ACCENT = RGBColor(0x1F, 0x29, 0x33), RGBColor(0x52, 0x60, 0x6D), RGBColor(0x1F, 0x4E, 0x79)
HEAD_FILL, FLAG_FILL, RULE = "E4EAF0", "FBEFD5", "C5CDD5"
FONT = "Arial"

f1 = lambda x: f"{x:.1f}"
fq = lambda x: f"{x:.1f}" if float(x * 2).is_integer() else f"{x:.2f}"
sgn = lambda x: ("+" if x >= 0.05 else "-" if x <= -0.05 else "") + f"{abs(x):.1f}"
line_no = lambda x: f"{x:.0f}" if float(x).is_integer() else f"{x}"
pct = lambda x: "n/a" if pd.isna(x) else f"{round(x * 100)}%"
ml = lambda x: "n/a" if pd.isna(x) else ("+" if x > 0 else "-") + f"{abs(round(x))}"
game = lambda r: f"{NICK[r['away']]} at {NICK[r['home']]}"
def score(an, ap, hn, hp, f=f1):
    return f"{hn} {f(hp)}, {an} {f(ap)}" if hp >= ap else f"{an} {f(ap)}, {hn} {f(hp)}"

doc = docx.Document()
for z in doc.settings.element.findall(qn("w:zoom")):
    z.set(qn("w:percent"), "100")
st = doc.styles["Normal"]; st.font.name = FONT; st.font.size = Pt(10.5); st.font.color.rgb = INK
st.paragraph_format.space_after = Pt(7); st.paragraph_format.line_spacing = 1.2
for name, size, color in [("Heading 1", 14, ACCENT), ("Heading 2", 11.5, INK)]:
    h = doc.styles[name]; h.font.name = FONT; h.font.size = Pt(size); h.font.bold = True; h.font.color.rgb = color
    for attr in ("w:asciiTheme", "w:hAnsiTheme", "w:eastAsiaTheme", "w:cstheme"):     # theme fonts would override Arial
        h.element.rPr.rFonts.attrib.pop(qn(attr), None)
    h.paragraph_format.space_before = Pt(16 if name == "Heading 1" else 11); h.paragraph_format.space_after = Pt(6)
    h.paragraph_format.keep_with_next = True

def set_page(section, landscape=False):
    w, h = Inches(8.5), Inches(11)
    section.orientation = WD_ORIENT.LANDSCAPE if landscape else WD_ORIENT.PORTRAIT
    section.page_width, section.page_height = (h, w) if landscape else (w, h)
    section.left_margin = section.right_margin = Inches(1)
    section.top_margin = section.bottom_margin = Inches(0.85 if landscape else 0.9)

def footer(section):
    section.footer.is_linked_to_previous = False
    p = section.footer.paragraphs[0]; p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for old in list(p.runs): old._r.getparent().remove(old._r)
    run = p.add_run(f"Hidden Value, Week {WEEK} projections   |   Page "); run.font.size = Pt(8); run.font.color.rgb = MUTED
    r2 = p.add_run(); r2.font.size = Pt(8); r2.font.color.rgb = MUTED
    for kind, text in [("begin", None), (None, "PAGE"), ("end", None)]:
        if kind:
            el = OxmlElement("w:fldChar"); el.set(qn("w:fldCharType"), kind)
        else:
            el = OxmlElement("w:instrText"); el.set(qn("xml:space"), "preserve"); el.text = text
        r2._r.append(el)

def para(text="", size=None, color=None, bold_lead=None, style=None, after=None, indent=None):
    """bold_lead: text printed in bold before the rest."""
    p = doc.add_paragraph(style=style)
    if bold_lead:
        r = p.add_run(bold_lead); r.bold = True
        if size: r.font.size = Pt(size)
    r = p.add_run(text)
    if size: r.font.size = Pt(size)
    if color is not None: r.font.color.rgb = color
    if after is not None: p.paragraph_format.space_after = Pt(after)
    if indent:
        p.paragraph_format.left_indent = Inches(indent); p.paragraph_format.first_line_indent = Inches(-0.22)
    return p

def bullet(text, lead=None):
    return para(text, bold_lead=lead, style="List Bullet", after=4)

def numbered(items):
    for i, (lead, text) in enumerate(items, 1):
        para(text, bold_lead=f"{i}.  {lead}", after=4, indent=0.4)

def shade(cell, fill):
    tcPr = cell._tc.get_or_add_tcPr(); shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), fill); tcPr.append(shd)

def table(widths, header, body, size=9, center=(), flag=None, bold_rows=(), bold_cols=(), keep=False):
    """widths in inches. body: list of lists; a cell may be a list of lines. flag: row indexes to shade.
    keep=True holds a short table together on one page."""
    t = doc.add_table(rows=1 + len(body), cols=len(widths)); t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False
    for ci, w in enumerate(widths):
        t.columns[ci].width = Inches(w)
    borders = OxmlElement("w:tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        el = OxmlElement(f"w:{edge}"); el.set(qn("w:val"), "single"); el.set(qn("w:sz"), "4"); el.set(qn("w:color"), RULE); borders.append(el)
    anchor = next((c for c in t._tbl.tblPr if c.tag in (qn("w:shd"), qn("w:tblLayout"), qn("w:tblCellMar"), qn("w:tblLook"))), None)
    if anchor is not None: anchor.addprevious(borders)          # Word expects table borders before these settings
    else: t._tbl.tblPr.append(borders)
    hdr = OxmlElement("w:tblHeader"); hdr.set(qn("w:val"), "true"); t.rows[0]._tr.get_or_add_trPr().append(hdr)
    for ri, cells in enumerate([header] + body):
        cant = OxmlElement("w:cantSplit"); cant.set(qn("w:val"), "true"); t.rows[ri]._tr.get_or_add_trPr().append(cant)
        for ci, content in enumerate(cells):
            c = t.cell(ri, ci); c.width = Inches(widths[ci])
            lines = content if isinstance(content, list) else [content]
            for li, text in enumerate(lines):
                p = c.paragraphs[0] if li == 0 else c.add_paragraph()
                p.paragraph_format.space_after = Pt(0); p.paragraph_format.space_before = Pt(0); p.paragraph_format.line_spacing = 1.05
                p.paragraph_format.keep_with_next = bool(keep and ri < len(body))
                if ci in center: p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                r = p.add_run(str(text)); r.font.size = Pt(size)
                r.bold = ri == 0 or (ri - 1) in bold_rows or (ci in bold_cols and ri > 0)
            if ri == 0: shade(c, HEAD_FILL)
            elif flag is not None and (ri - 1) in flag: shade(c, FLAG_FILL)
            tcPr = c._tc.get_or_add_tcPr(); mar = OxmlElement("w:tcMar")
            for side, v in (("top", 55), ("left", 100), ("bottom", 55), ("right", 100)):
                el = OxmlElement(f"w:{side}"); el.set(qn("w:w"), str(v)); el.set(qn("w:type"), "dxa"); mar.append(el)
            tcPr.append(mar)
    doc.add_paragraph().paragraph_format.space_after = Pt(2)
    return t

n_open, n_diff = len(open_games), int((~open_games["same_side"]).sum())
_ts = pd.to_datetime(P["saved_at"].iloc[0])
SAVED = f"{_ts.strftime('%B')} {_ts.day}, {_ts.year} at {_ts.strftime('%I:%M %p').lstrip('0')}"
set_page(doc.sections[0]); footer(doc.sections[0])

# ----- title -----
p = doc.add_paragraph(); r = p.add_run(f"Hidden Value: Week {WEEK} Points Projections"); r.bold = True; r.font.size = Pt(20); r.font.color.rgb = ACCENT
p.paragraph_format.space_after = Pt(3)
para("How the projections are built, the projected scores, and how they compare with the betting line", size=11.5, color=MUTED, after=3)
p = para(f"{SEASON} NFL season, Week {WEEK}. Projections saved {SAVED}.", size=9.5, color=MUTED, after=12)
bdr = OxmlElement("w:pBdr"); bot = OxmlElement("w:bottom")
for k, v in (("val", "single"), ("sz", "8"), ("space", "8"), ("color", "1F4E79")): bot.set(qn(f"w:{k}"), v)
bdr.append(bot)
_pPr = p._p.get_or_add_pPr(); _sp = _pPr.find(qn("w:spacing"))
if _sp is not None: _sp.addprevious(bdr)                           # Word expects the border before spacing
else: _pPr.append(bdr)

# ----- summary -----
doc.add_heading("Summary", 1)
bullet("Projects each team's points from four things: its own points per drive, the opponent's points per drive allowed, home field, and who is playing quarterback.", "What it does. ")
bullet(f"The model and the betting line pick a different winner in {n_diff} of the {n_open} games projected in advance. "
       f"The model's totals average {f1(open_games['total'].mean())} points a game against the line's {f1(open_games['line_total'].mean())}.", "How it compares with the betting line. ")
bullet("The model is new, so it has little track record. When the same formula was replayed on 2021 to 2025 games (a backtest, explained below), "
       "it missed a team's score by 7.5 points on average. The betting market missed by 7.25. A single team's score is mostly noise, and the market is the better forecaster.", "How accurate to expect. ")
if len(graded):
    miss = graded[MODELS].sub(graded["actual"], axis=0).abs().mean()
    bullet(f"{len(graded)} team scores graded through Week {int(graded['week'].max())}. Average miss: base model {miss['Base model']:.2f}, betting line {miss['Betting line']:.2f}.", "Record so far. ")
if A is not None:
    bullet("A trial Advanced version adds quarterback history, missing starters and wind. Its projections are shown next to the base model on the comparison pages.", "Trial version. ")

if NOTES:
    doc.add_heading("Notes for this week", 1)
    for n in NOTES: bullet(n)

# ----- how it is built -----
doc.add_heading("How a projection is built", 1)
para("Every team's projected points come from the same four inputs:")
numbered([("Its offense. ", "Points per drive this season, counting a touchdown drive as 7 and a field goal drive as 3."),
          ("The opponent's defense. ", "Points per drive that defense has allowed."),
          ("Home field. ", "A fixed edge for the home team and the same amount taken off the road team."),
          ("Quarterback continuity. ", "A penalty when the expected starter has not run most of the team's drives this season.")])
para("Two adjustments are made before those inputs are used. First, each rating is pulled toward the league average by adding 20 league-average drives, so a team with few drives is not taken at face value. "
     "Second, the offense rating leans on the expected starting quarterback: his drives count fully and drives run by any other quarterback count at 25%.")
doc.add_heading("The formula", 2)
table([1.7, 1.1, 3.7], ["Piece", "Weight", "What it does"], [
    ["Base", "4.41", "Starting points for every team."],
    ["Offense rating", "x 6.22", "The team's adjusted points per drive. The largest piece."],
    ["Opponent's defense rating", "x 2.47", "Adjusted points per drive the opponent allows."],
    ["Home field", "+1.15 or -1.15", "Added at home, subtracted on the road, zero at a neutral site."],
    ["New quarterback", "x -3.38", "Multiplied by the share of the team's drives the expected starter has not run."]], keep=True)
para("The weights were not chosen by hand. They come from a backtest on the 2021 to 2025 seasons, explained in the next section. None of them were set or tuned on this season's results.")

if A is not None and len(A[~A["already_played"].astype(bool)]):
    ex = A[~A["already_played"].astype(bool)].sort_values("base", ascending=False).iloc[0]     # highest base projection this week
    where = "at home" if ex["home"] > 0 else "on the road" if ex["home"] < 0 else "neutral site"
    doc.add_heading(f"Worked example: {NICK[ex['team']]} against {NICK[ex['opp']]}", 2)
    table([2.2, 2.9, 1.4], ["Piece", "Calculation", "Points"], [
        ["Base", "constant", f"{COEF['const']:.2f}"],
        [f"{NICK[ex['team']]} offense", f"{COEF['off']} x {ex['off']:.2f}", f"{COEF['off'] * ex['off']:.2f}"],
        [f"{NICK[ex['opp']]} defense", f"{COEF['deff']} x {ex['deff']:.2f}", f"{COEF['deff'] * ex['deff']:.2f}"],
        ["Home field", where, f"{COEF['home'] * ex['home']:.2f}"],
        ["New quarterback", f"{ex['qb']} has run {round((1 - ex['new_qb']) * 100)}% of the drives", f"{COEF['new_qb'] * ex['new_qb'] + 0.0:.2f}"],
        [f"{NICK[ex['team']]} projected points", "", f"{ex['base']:.1f}"]], bold_rows=[5], center=(2,), keep=True)

# ----- backtest -----
doc.add_heading("Where the numbers come from: the backtest", 1)
para("This model is new. Nobody was running it in 2021, so it has no long track record. A backtest is the next best thing: it replays past seasons as if the model had existed then.")
para("For every week from Week 4 through Week 18 of the 2021 to 2025 seasons, the same three steps were run:")
numbered([("Use only what was known at the time. ", "Ratings were built from drives in earlier weeks of that season and nothing later."),
          ("Project every game that week. ", "With the same formula used this season."),
          ("Compare with the real final scores. ", "The difference between projected and actual points is the miss.")])
para("That gave 2,238 team scores across 1,119 games to check the formula against.")
doc.add_heading("What the backtest found", 2)
bt = [r for r in BACKTEST if A is not None or not r[0].startswith("Advanced")]
table([2.9, 1.25, 1.35, 1.0], ["Method", "Average miss per team", "Average miss on game total", "Picks winner"], [list(r) for r in bt],
      center=(1, 2, 3), bold_rows=[i for i, r in enumerate(bt) if r[0] == "Base model"], keep=True)
para("The drive statistics alone do no better than plain points for and against. The gain comes from knowing who is at quarterback, and the market is still ahead. "
     "No version beat the point spread in the backtest (about 49% to 50%), so this is a forecast, not a betting edge.")
doc.add_heading("What a backtest is not", 2)
para("It is a simulation, not a track record. Backtests usually look a little better than real results, because the formula was shaped on those same seasons. "
     "To limit that, each season was projected using weights fitted only on the other four, but that does not remove the effect entirely. "
     "The projections saved in advance and graded against final scores are the numbers that count.")

# ----- what it accounts for -----
doc.add_heading("What the model accounts for", 1)
adv_note = " The trial Advanced version counts regular starters who are out." if A is not None else ""
wx_note = " The trial Advanced version uses wind above 10 mph in outdoor games." if A is not None else ""
table([1.9, 0.9, 3.7], ["Factor", "In the base model?", "How"], [
    ["Offense", "Yes", "Points per drive, pulled toward league average."],
    ["Opposing defense", "Yes", "Points per drive allowed. It gets less weight than offense because, in the backtest, a team's offense predicted its next game about three times better than the opposing defense did."],
    ["Home field", "Yes", "1.15 points each way."],
    ["Quarterback injuries and changes", "Yes", "The offense rating favors the expected starter's drives, and a new starter takes a direct penalty."],
    ["Other injuries", "No", "A missing receiver, lineman or defender changes nothing." + adv_note],
    ["Strength of opponents faced", "No", "Ratings are not adjusted for schedule. That adjustment was tested and did not improve accuracy."],
    ["Weather, rest, travel", "No", "Not used." + wx_note]], keep=True)

# ----- projected scores -----
doc.add_page_break()
doc.add_heading(f"Week {WEEK} projected scores", 1)
para("Quarterbacks are listed road team first. They are the starters the model assumed.")
body = []
for _, r in G.iterrows():
    name = [game(r), "(already played when saved)"] if r["played_when_saved"] else game(r)
    win = NICK[r["home"]] if r["margin"] > 0 else NICK[r["away"]]
    body.append([name, f"{r['away_qb']} / {r['home_qb']}", score(NICK[r["away"]], r["away_pts"], NICK[r["home"]], r["home_pts"]), f"{win} by {f1(abs(r['margin']))}", f1(r["total"])])
table([1.65, 1.6, 1.9, 0.95, 0.5], ["Game", "Quarterbacks assumed", "Projected score", "Model pick", "Total"], body, size=8.5, center=(4,))
toss = [game(r) for _, r in open_games.iterrows() if abs(r["margin"]) < 1.05]
if toss:
    para(f"A projected margin of about a point or less is a toss-up, not a pick. That covers {len(toss)} game{'s' if len(toss) != 1 else ''}: {'; '.join(toss)}.")
late = G[G["played_when_saved"]]
for _, r in late.iterrows():
    if pd.notna(r["home_score"]):
        para(f"{game(r)} had already been played when the projections were saved, so it is not graded. Final: {score(NICK[r['away']], r['away_score'], NICK[r['home']], r['home_score'], lambda v: f'{v:.0f}')}. "
             f"The model had {score(NICK[r['away']], r['away_pts'], NICK[r['home']], r['home_pts'])}.")

# ----- comparison pages (landscape) -----
sec = doc.add_section(WD_SECTION.NEW_PAGE); set_page(sec, landscape=True); footer(sec)
doc.add_heading("Model against the betting line", 1)
para("Each row is read from the side of the betting favorite, so the model and the line can be compared number for number. "
     "Lines are from the nflverse schedule file at the time this report was run. In Week 5 they matched DraftKings on every game checked.")
doc.add_heading("How to read the tables", 2)
bullet("The margin the favorite is expected to win by. -7.5 means favored by 7.5. A plus number in the model column means the model has that team losing by that much.", "Spread. ")
bullet("Betting line: both moneylines turned into percentages, with the sportsbook's cut removed. Model: the projected margin turned into a percentage using the backtest.", "Win chance. ")
bullet("The price to take a team to win outright. -425 means risking 425 to win 100. +116 means risking 100 to win 116. The model's fair moneyline is the price that matches its win chance exactly, with no sportsbook cut.", "Moneyline. ")
bullet("Points for one team, and points for both teams combined. The line's team totals are worked out from its spread and total.", "Team total and game total. ")
bullet("A model spread within about a point either way is a coin flip. In the backtest, teams the model favored by less than a point won 46% of the time.", "Toss-ups. ")

doc.add_page_break()
doc.add_heading("Who wins: spread, win chance and moneyline", 2)
og = open_games.reset_index(drop=True)
body = [[game(r), NICK[r["fav"]], f"-{line_no(r['line_spread'])}", sgn(-r["fav_margin"]), pct(r["line_p"]), pct(r["model_p"]),
         ml(r["line_fav_ml"]), ml(r["model_fair_ml"]), "Yes" if r["same_side"] else "No"] for _, r in og.iterrows()]
table([1.7, 1.15, 0.85, 0.85, 0.95, 0.95, 0.95, 0.95, 0.65],
      ["Game", ["Betting", "favorite"], ["Spread", "Line"], ["Spread", "Model"], ["Win chance", "Line"], ["Win chance", "Model"],
       ["Moneyline", "Line"], ["Moneyline", "Model fair"], ["Same", "side?"]], body,
      center=(2, 3, 4, 5, 6, 7, 8), flag=[i for i, s_ in enumerate(og["same_side"]) if not s_], bold_cols=(1,))
para(f"Every number is for the team in the favorite column. Shaded rows are the {n_diff} game{'s' if n_diff != 1 else ''} where the model has the other team winning.", size=9, color=MUTED)

doc.add_page_break()
doc.add_heading("Points: team totals and game total", 2)
body = [[game(r), f"{NICK[r['away']]} {fq(r['line_away'])}, {NICK[r['home']]} {fq(r['line_home'])}",
         f"{NICK[r['away']]} {f1(r['away_pts'])}, {NICK[r['home']]} {f1(r['home_pts'])}", line_no(r["line_total"]), f1(r["total"]), sgn(r["total_gap"])] for _, r in og.iterrows()]
table([1.9, 2.4, 2.4, 0.75, 0.75, 0.8], ["Game", ["Team totals", "Line (from spread and total)"], ["Team totals", "Model"], ["Game total", "Line"], ["Game total", "Model"], ["Model", "minus line"]],
      body, center=(3, 4, 5))
para("Team totals are listed road team first in both columns.", size=9, color=MUTED)

if A is not None:
    doc.add_page_break()
    doc.add_heading("Three versions of the model", 2)
    para("Base is the model described in this document. Advanced adds quarterback history, regular starters who are out, and wind. Advanced + market also starts each team from where the betting line had it in earlier games. "
         "In the backtest the two Advanced versions missed by 7.42 and 7.38 points per team against 7.50 for Base. They are being tracked to see whether that holds up.")
    a_open = A[~A["already_played"].astype(bool)]
    hm_, aw_ = a_open[a_open["is_home"]].set_index("game_id"), a_open[~a_open["is_home"]].set_index("game_id")
    body = []
    for _, r in og.iterrows():
        if r["game_id"] not in hm_.index or r["game_id"] not in aw_.index:
            continue
        h, a = hm_.loc[r["game_id"]], aw_.loc[r["game_id"]]
        an, hn = NICK[r["away"]], NICK[r["home"]]
        body.append([game(r)] + [f"{an} {f1(a[m])}, {hn} {f1(h[m])}" for m in ("base", "adv", "adv_mkt")]
                    + [f"{an} {fq(r['line_away'])}, {hn} {fq(r['line_home'])}"])
    table([1.75, 1.81, 1.81, 1.81, 1.82], ["Game", "Base", "Advanced", "Advanced + market", "Betting line"], body, size=8.5)
    para("Scores are listed road team first in every column.", size=9, color=MUTED)
    adj = a_open.assign(q=a_open["pts_qb_history"].abs(), i=a_open["pts_injuries"].abs())
    top_q, top_i = adj.sort_values("q", ascending=False).head(3), adj[adj["i"] > 0.05].sort_values("i", ascending=False).head(3)
    para("Largest quarterback-history adjustments: " + "; ".join(f"{NICK[r['team']]} ({r['qb']}) {sgn(r['pts_qb_history'])}" for _, r in top_q.iterrows()) + ".", size=9, color=MUTED)
    if len(top_i):
        para("Largest missing-starter adjustments: " + "; ".join(f"{NICK[r['team']]} {sgn(r['pts_injuries'])}" for _, r in top_i.iterrows()) + ".", size=9, color=MUTED)
    windy = a_open[a_open["wind_over"] > 0]
    if len(windy):
        para("Wind applied in: " + "; ".join(sorted({game(og[og['game_id'] == g_].iloc[0]) for g_ in windy["game_id"] if (og['game_id'] == g_).any()})) + ".", size=9, color=MUTED)

# ----- back to portrait -----
sec = doc.add_section(WD_SECTION.NEW_PAGE); set_page(sec); footer(sec)
doc.add_heading("Biggest gaps in the projected margin", 2)
for _, r in og.reindex(og["margin_gap"].abs().sort_values(ascending=False).index).head(5).iterrows():
    win = NICK[r["home"]] if r["margin"] > 0 else NICK[r["away"]]
    bullet(f"the model has {win} by {f1(abs(r['margin']))}, the line has {NICK[r['fav']]} by {line_no(r['line_spread'])}. A gap of {f1(abs(r['margin_gap']))} points.", f"{game(r)}: ")
doc.add_heading("Biggest gaps in the projected total", 2)
for _, r in og.reindex(og["total_gap"].abs().sort_values(ascending=False).index).head(4).iterrows():
    bullet(f"model total {f1(r['total'])}, line {line_no(r['line_total'])} (model is {'higher' if r['total_gap'] > 0 else 'lower'} by {f1(abs(r['total_gap']))}).", f"{game(r)}: ")
para(f"In the backtest the market picked 68% of winners and the base model 62%. When the two disagreed, the market's team won about two times in three.")

doc.add_heading("How the projections are graded", 1)
numbered([("They are saved before kickoff. ", "The projection notebook writes its numbers to a file and refuses to overwrite it, so nothing can be adjusted after the fact."),
          ("Each team's projection is compared with its final score. ", "The measure is the average miss per team, in points. Lower is better."),
          ("The betting line is graded on the same games. ", "Using the team totals implied by the spread and total that were saved with the projections."),
          ("Winners picked are counted. ", "Out of the games projected in advance.")])
doc.add_heading("Results so far", 2)
if len(graded):
    miss = graded[MODELS].sub(graded["actual"], axis=0).abs().mean()
    hm_, aw_ = graded[graded["is_home"]].set_index("game_id"), graded[~graded["is_home"]].set_index("game_id")
    both = hm_.index.intersection(aw_.index); real = hm_.loc[both, "actual"] - aw_.loc[both, "actual"]
    body = [[m, f"{miss[m]:.2f}", f"{int((np.sign(hm_.loc[both, m] - aw_.loc[both, m]) == np.sign(real)).sum())} of {len(both)}"] for m in MODELS]
    table([2.6, 1.6, 1.6], ["Version", "Average miss per team", "Winners picked"], body, center=(1, 2), keep=True)
    wks = sorted(graded["week"].unique().tolist())
    para(f"{len(graded)} team scores graded, from week{'s' if len(wks) != 1 else ''} {', '.join(map(str, wks))}." +
         (" Under about 100 team scores this comparison is mostly noise, so it is too early to draw conclusions." if len(graded) < 100 else ""))
else:
    para("No game that was projected in advance has a final score yet. This table fills in once the games are played and this notebook is run again.")

doc.add_heading("Sources", 1)
bullet("Play-by-play, schedule, betting lines and final scores: nflverse data, loaded with nflreadpy.")
bullet(f"Projections: {os.path.basename(base_files[WEEK])}" + (f" and {os.path.basename(adv_files[WEEK])}" if A is not None else "") + f", saved {SAVED}.")
bullet("Backtest figures: the appendix of the projection notebooks, 2021 to 2025 seasons.")

out_dir = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\WEEKLY PREDICTIONS"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, f"Hidden_Value_Week{WEEK:02d}_Projections.docx")
try:
    doc.save(out_path)
    print("saved", out_path)
except PermissionError:
    print(">> Could not save. Close", os.path.basename(out_path), "in Word and run this cell again.")

saved C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\WEEKLY PREDICTIONS\Hidden_Value_Week09_Projections.docx


In [11]:
# Week 9 "what if": grade every game with a final score, including ones saved after kickoff
WK = 9
p = pd.read_csv(base_files[WK])
a = pd.read_csv(adv_files[WK]) if WK in adv_files else None

rows = []
for r in p.itertuples():
    s = S.loc[r.game_id]
    if pd.isna(s["home_score"]):
        continue
    for team, pts, actual, mk in [(r.home, r.home_pts, s["home_score"], r.mkt_total / 2 + r.mkt_home_margin / 2),
                                  (r.away, r.away_pts, s["away_score"], r.mkt_total / 2 - r.mkt_home_margin / 2)]:
        row = {"game_id": r.game_id, "team": team, "is_home": team == r.home, "actual": actual,
               "Base model": pts, "Betting line": mk, "late_save": bool(r.already_played)}
        if a is not None:
            m = a[(a["game_id"] == r.game_id) & (a["team"] == team)]
            if len(m):
                row["Advanced"], row["Advanced + market"] = m["adv"].iloc[0], m["adv_mkt"].iloc[0]
        rows.append(row)
g9 = pd.DataFrame(rows)
models = [m for m in ["Base model", "Advanced", "Advanced + market", "Betting line"] if m in g9.columns]

# average miss per team
print(f"Week {WK}: {len(g9)} team scores, {g9.groupby('game_id').ngroups} games\n")
print("Average miss per team:")
print(g9[models].sub(g9["actual"], axis=0).abs().mean().round(2).to_string(), "\n")

# winners picked
home, away = g9[g9["is_home"]].set_index("game_id"), g9[~g9["is_home"]].set_index("game_id")
actual_margin = home["actual"] - away["actual"]
print("Winners picked:")
for m in models:
    right = (np.sign(home[m] - away[m]) == np.sign(actual_margin)).sum()
    print(f"  {m:<18} {right} of {len(actual_margin)}")

# game-by-game
show = home[["team"]].rename(columns={"team": "home"}).join(away[["team"]].rename(columns={"team": "away"}))
for m in models + ["actual"]:
    show[m] = away[m].round(1).astype(str) + " - " + home[m].round(1).astype(str)
print("\nScores, road team first:")
print(show.to_string())

Week 9: 28 team scores, 14 games

Average miss per team:
Base model           7.24
Advanced             7.37
Advanced + market    7.30
Betting line         6.55 

Winners picked:
  Base model         12 of 14
  Advanced           12 of 14
  Advanced + market  11 of 14
  Betting line       13 of 14

Scores, road team first:
                home away   Base model     Advanced Advanced + market Betting line       actual
game_id                                                                                        
2026_09_JAX_BAL  BAL  JAX  25.9 - 19.0  26.2 - 18.9       25.8 - 19.5  24.5 - 16.0  33.0 - 10.0
2026_09_CIN_ATL  ATL  CIN  27.0 - 23.3  26.6 - 23.6       27.4 - 22.6  27.8 - 24.8  34.0 - 17.0
2026_09_DEN_CAR  CAR  DEN  21.9 - 24.9  25.8 - 25.7       25.6 - 26.0  23.8 - 20.8  26.0 - 21.0
2026_09_DAL_IND  IND  DAL  27.0 - 25.0  30.3 - 24.7       30.4 - 24.1  28.0 - 23.5  27.0 - 23.0
2026_09_NYJ_KC    KC  NYJ  22.1 - 26.4  22.4 - 26.1       21.3 - 25.7  19.0 - 28.5  12.0 - 35.0
202